# செலவு கோரிக்கை பகுப்பாய்வு

இந்த நோட்புக், உள்ளூர் ரசீதுப் படங்களை பயன்படுத்தி பயணச் செலவுகளை செயலாக்குவதற்கான பிளாக்கின்களைப் பயன்படுத்தும் முகவர்களை உருவாக்குவது எப்படி என்பதை காட்சி காட்டுகிறது, செலவு கோரிக்கை மின்னஞ்சலை உருவாக்குகிறது மற்றும் ஒரு பட்டைச் சாட்டைப் பயன்படுத்தி செலவு தரவுகளை காட்சிப்படுத்துகிறது. முகவர்கள் பணிப் பொருளின் அடிப்படையில் செயல்பாடுகளை தானாகத் தேர்ந்தெடுப்பார்கள்.

படிகள்:
1. OCR முகவர் உள்ளூர் ரசீதுப் படத்தை செயலாக்கி பயணச் செலவு தரவுகளை எடுத்துக் கொள்கிறார்.
2. மின்னஞ்சல் முகவர் செலவு கோரிக்கை மின்னஞ்சலை உருவாக்குகிறார்.

### ஒரு பயணச் செலவு சின்னாரியோவின் உதாரணம்:
நீங்கள் வேறு ஒரு நகரத்தில் உள்ள வணிகக் கூட்டத்திற்குப் பயணிக்கும் ஒரு ஊழியர் என்று கற்பனை செய்யுங்கள். உங்கள் நிறுவனம் அனைத்து நியாயமான பயணச் செலவுகளையும் திரும்பச் செலுத்தும் கொள்கையை கொண்டுள்ளது. இங்கே சாத்தியமான பயணச் செலவுகளின் விரிவுரை:
- போக்குவரத்து:
உங்கள் வீட்டு நகரம் முதல் இலக்கு நகரத்துக்குள் சுற்றுலா விமானச் செலவு.
விமான நிலையத்திற்கு செல்லும் மற்றும் வரும் டாக்ஸி அல்லது சவாரி சேவைகள்.
இலக்கு நகரத்தில் உள்ள உள்ளூர் போக்குவரத்து (பொது போக்குவரத்து, வாடகை கார்கள் அல்லது டாக்ஸிகள் போன்றவை).

- தங்குமிடம்:
கூட்டம் நடைபெறும் இடத்திற்கு அருகிலான நடுத்தர அளவிலான வணிக ஹோட்டலில் மூன்று இரவுகள் தங்குதல்.

- உணவுகள்:
நிறுவனத்தின் தினசரி குறுகிய கட்டண நெறிமுறையின் அடிப்படையில் காலை உணவு, மதிய உணவு மற்றும் இரவு உணவுக்கான தினசரி உணவு அனுமதி.

- பல்வேறு செலவுகள்:
விமான நிலையத்தில் நிறுத்தும் கட்டணங்கள்.
ஹோட்டலில் இணைய உபயோக கட்டணங்கள்.
அஞ்சலிகள் அல்லது சிறிய சேவை கட்டணங்கள்.

- ஆவணங்கள்:
நீங்கள் அனைத்து ரசீதுக்களும் (விமானங்கள், டாக்ஸிகள், ஹோட்டல், உணவுகள், மற்றும் பிற) மற்றும் ஒரு நிறைவு செய்யப்பட்ட செலவு அறிக்கை மீளச்செலுத்துதற்காக சமர்ப்பிக்கிறீர்கள்.


## தேவையான நூலகங்களை இறக்குமதி செய்யவும்

நோட்புக்காக தேவையான நூலகங்கள் மற்றும் தொகுதிகளை இறக்குமதி செய்யவும்.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## செலவு மாதிரிகளை வரையறு

 தனிப்பட்ட செலவுகளுக்கான Pydantic மாதிரியும் பயனர் கேள்வியினை கட்டமைக்கப்பட்ட செலவு தரவாக மாற்ற ஒரு ExpenseFormatter வகுப்பையும் உருவாக்கவும்.

 ஒவ்வொரு செலவுக்கும் பின்வரும் வடிவமைப்பில் பிரதிநிதித்துவம் செய்யப்படும்:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## கருவிகள் வரையறு - மின்னஞ்சலை உருவாக்குதல்

ஒரு செலவுக் கோரிக்கை சமர்ப்பிக்க மின்னஞ்சல் உருவாக்க கருவி செயல்பாட்டை உருவாக்கவும்.
- இந்த கருவி Microsoft Agent Framework இலிருந்து `@tool` அலங்கரிப்பை பயன்படுத்துகிறது.
- செலவுகளின் மொத்தத்தையும் கணக்கிட்டு விவரங்களை மின்னஞ்சல் உடலாக வடிவமைக்கிறது.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# ரசீதின் படங்களிலிருந்து பயணச் செலவுகளை எடுக்க ஒரு கருவி

ரசீதின் படங்களிலிருந்து பயணச் செலவுகளை எடுக்க ஒரு கருவி செயல்பாட்டை உருவாக்கவும்.
- இந்த கருவி Microsoft Agent Framework இன் `@tool` அலங்காரியை பயன்படுத்துகிறது.
- இது ரசீதின் படத்தை வாசித்து, அதை base64 ஆக குறியாக்கி, ஆய்வு செய்ய காரியகருவிக்கு தரவுப் URIஐ வழங்குகிறது.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## செலவுகளை செயலாக்குதல்

ஏஜெண்ட்களை வரையறு மற்றும் அவற்றை வரிசைப்படுத்தப்பட்ட வேலைப்ப流程ம் ஒன்றாக `WorkflowBuilder` பயன்படுத்தி இணைக்கவும்.
- OCR ஏஜெண்ட் ரசீது படத்திலிருந்து கட்டமைக்கப்பட்ட செலவு தரவை `load_receipt_image` கருவியைப் பயன்படுத்தி எடுக்கிறது.
- மின்னஞ்சல் ஏஜெண்ட் எடுக்கப்பட்ட தரவை எடுத்துக் கொண்டு `generate_expense_email` கருவியைப் பயன்படுத்தி தொழில்முறை செலவு கோரிக்கை மின்னஞ்சலை உருவாக்குகிறது.
- `WorkflowBuilder` மற்றும் `add_edge` மூலம் வரிசைப்படுத்தப்பட்ட குழாய் உருவாக்கப்படுகிறது: OCR ஏஜெண்ட் → மின்னஞ்சல் ஏஜெண்ட்.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

தொடர்ச்சியான வேலைப்ப 흐ி வையை உருவாக்கி, ரசீது படத்தை செயல்படுத்தவும் செலவுத் தீர்பு மின்னஞ்சலை உருவாக்கவும் அதை இயக்கவும்.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
